# 14 Stabilitas Partisi


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next(
    (p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()),
    None
)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')

LIB_DIR = PAPER_ROOT/'LIB'
OUT_DIR = stage_dir('14_Stabilitas_Partisi')
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el

MEMBER_FILE = artifact_path('semantic_stability_member_comparison.csv','14_Stabilitas_Partisi')
REFERENCE_FILE = artifact_path('04_carekm_reference_labels.csv','14_Stabilitas_Partisi')
ROBUST_FILE = artifact_path('06_robustness_raw.csv','14_Stabilitas_Partisi')

if not MEMBER_FILE.is_file():
    raise FileNotFoundError(
        'semantic_stability_member_comparison.csv tidak ditemukan. '
        'Jalankan 13_Stabilitas_Semantik.ipynb terlebih dahulu.'
    )
if not REFERENCE_FILE.is_file():
    raise FileNotFoundError(
        '04_carekm_reference_labels.csv tidak ditemukan.'
    )

member = pd.read_csv(
    MEMBER_FILE,
    dtype={'user':str,'sample_hash':str},
)
reference = pd.read_csv(
    REFERENCE_FILE,
    dtype={'user':str},
)

required = {'rep','sample_hash','user','cluster_id_resample'}
if not required.issubset(member.columns):
    raise RuntimeError(
        f'Kolom member comparison tidak lengkap: {sorted(required-set(member.columns))}'
    )

member['rep'] = pd.to_numeric(member['rep'], errors='raise').astype(int)
member['cluster_id_resample'] = pd.to_numeric(
    member['cluster_id_resample'], errors='raise'
).astype(int)
reference['cluster_id'] = pd.to_numeric(
    reference['cluster_id'], errors='raise'
).astype(int)

if reference['user'].duplicated().any():
    raise RuntimeError('Reference labels memiliki user duplikat.')

ref_map = reference.set_index('user')['cluster_id']
member['cluster_id_reference'] = member['user'].map(ref_map)

if member['cluster_id_reference'].isna().any():
    raise RuntimeError('Ada user resample yang tidak ditemukan pada full-data reference.')

member['cluster_id_reference'] = member['cluster_id_reference'].astype(int)

if member['rep'].nunique() != 30:
    raise RuntimeError(
        f'Diharapkan 30 resamples, ditemukan {member["rep"].nunique()}.'
    )

print('PAPER_ROOT:', PAPER_ROOT)
print('Member-resample rows:', len(member))
print('Repetitions:', member['rep'].nunique())

# Optional exact sample-hash cross-check with Notebook 06.
hash_check = []

if ROBUST_FILE.is_file():
    robust = pd.read_csv(ROBUST_FILE, dtype={'sample_hash':str})
    robust['rep'] = pd.to_numeric(robust['rep'], errors='raise').astype(int)

    robust_hash = (
        robust.groupby('rep')['sample_hash']
        .agg(lambda x: sorted(set(x.astype(str))))
    )

    if not robust_hash.apply(len).eq(1).all():
        raise RuntimeError('Notebook 06 memiliki lebih dari satu sample_hash pada rep yang sama.')

    robust_hash = robust_hash.apply(lambda x: x[0])

    for rep in sorted(member['rep'].unique()):
        observed = sorted(
            member.loc[member['rep'].eq(rep),'sample_hash']
            .astype(str).unique()
        )
        if len(observed) != 1:
            raise RuntimeError(f'Rep {rep} memiliki lebih dari satu sample_hash.')

        expected = str(robust_hash.loc[rep])
        hash_check.append({
            'rep':int(rep),
            'semantic_stability_hash':observed[0],
            'robustness_notebook06_hash':expected,
            'match':observed[0] == expected,
        })

    hash_check = pd.DataFrame(hash_check)
    if not hash_check['match'].all():
        raise RuntimeError('Sample hash tidak identik dengan Notebook 06.')

    print('Sample-hash verification against Notebook 06: PASS')
else:
    hash_check = pd.DataFrame(columns=[
        'rep','semantic_stability_hash',
        'robustness_notebook06_hash','match'
    ])
    print('06_robustness_raw.csv tidak ditemukan; hash cross-check dilewati.')

rows = []

for rep, g in member.groupby('rep', sort=True):
    y_ref = g['cluster_id_reference'].to_numpy(dtype=int)
    y_res = g['cluster_id_resample'].to_numpy(dtype=int)

    rows.append({
        'rep':int(rep),
        'sample_hash':str(g['sample_hash'].iloc[0]),
        'sample_n':int(len(g)),
        'reference_clusters_present':int(pd.Series(y_ref).nunique()),
        'resample_clusters_present':int(pd.Series(y_res).nunique()),
        'ari':float(adjusted_rand_score(y_ref, y_res)),
        'nmi':float(normalized_mutual_info_score(y_ref, y_res)),
        'ami':float(adjusted_mutual_info_score(y_ref, y_res)),
    })

metrics = pd.DataFrame(rows).sort_values('rep').reset_index(drop=True)

if len(metrics) != 30:
    raise RuntimeError('Metrics tidak memiliki tepat 30 repetitions.')

el.save_csv(
    metrics,
    artifact_path('partition_stability_resample_metrics.csv','14_Stabilitas_Partisi'),
    index=False
)

display(metrics)

N_BOOT = 10000
BOOT_SEED = 20261014

def bootstrap_mean_ci(values, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    boot = rng.choice(
        values,
        size=(N_BOOT, len(values)),
        replace=True
    ).mean(axis=1)
    lo, hi = np.quantile(boot, [0.025,0.975])
    return float(lo), float(hi)


summary_rows = []

for i, metric in enumerate(['ari','nmi','ami']):
    values = metrics[metric].to_numpy(dtype=float)
    lo, hi = bootstrap_mean_ci(values, BOOT_SEED+i)

    summary_rows.append({
        'metric':metric.upper(),
        'n_reps':int(len(values)),
        'mean':float(values.mean()),
        'sd':float(values.std(ddof=1)),
        'median':float(np.median(values)),
        'min':float(values.min()),
        'max':float(values.max()),
        'bootstrap_95ci_low':lo,
        'bootstrap_95ci_high':hi,
    })

summary = pd.DataFrame(summary_rows)

el.save_csv(
    summary,
    artifact_path('partition_stability_summary.csv','14_Stabilitas_Partisi'),
    index=False
)

if len(hash_check):
    el.save_csv(
        hash_check,
        artifact_path('partition_stability_sample_integrity.csv','14_Stabilitas_Partisi'),
        index=False
    )

display(summary)

validation = {
    'analysis':'CARE-KM partition stability across 30 matched 80% resamples',
    'reference_partition':'04_carekm_reference_labels.csv',
    'resample_partition_source':'semantic_stability_member_comparison.csv',
    'repetitions':30,
    'label_alignment_required':False,
    'hungarian_assignment_used':False,
    'metrics':['ARI','NMI','AMI'],
    'sample_hash_verified_against_notebook_06':bool(
        len(hash_check) and hash_check['match'].all()
    ),
    'outputs':[
        'partition_stability_resample_metrics.csv',
        'partition_stability_summary.csv',
        'partition_stability_sample_integrity.csv',
    ],
}

(artifact_path('partition_stability_validation.json','14_Stabilitas_Partisi')).write_text(
    json.dumps(validation, indent=2), encoding='utf-8'
)

print(json.dumps(validation, indent=2))